##### 1. Exploring the data
Class balance and missing values in the Scania APS training set.

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/raw/aps_failure_training_set.csv", skiprows = 20, na_values = "na")
test = pd.read_csv("../data/raw/aps_failure_test_set.csv", skiprows = 20, na_values = "na")

In [3]:
print(train.shape, test.shape)
train.head()

(60000, 171) (16000, 171)


,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


Training and test sets: 60,000 and 16,000 trucks, 170 anonymised sensor columns. The first 20 lines of each file are the licence; missing readings are coded as 'na'.

In [4]:
print(train["class"].value_counts())
print()
missing = train.isna().mean().sort_values(ascending=False)
print(missing.head(10).round(2))
print("Columns with no missing values:", (missing == 0).sum())

class
neg    59000
pos     1000
Name: count, dtype: int64

br_000    0.82
bq_000    0.81
bp_000    0.80
bo_000    0.77
ab_000    0.77
cr_000    0.77
bn_000    0.73
bm_000    0.66
bl_000    0.45
bk_000    0.38
dtype: float64
Columns with no missing values: 2


Failures are 1.7% of the training data, so accuracy is misleading. The evaluation uses recall, precision and Scania's cost metric (missed failure = 500, false alarm = 10).

In [5]:
print(test["class"].value_counts())
print()
cols = missing.head(10).index
print(train.groupby("class")[cols].apply(lambda g: g.isna().mean()).T.round(2))

class
neg    15625
pos      375
Name: count, dtype: int64

class    neg   pos
br_000  0.83  0.09
bq_000  0.82  0.09
bp_000  0.81  0.08
bo_000  0.78  0.08
ab_000  0.77  0.77
cr_000  0.77  0.77
bn_000  0.74  0.07
bm_000  0.67  0.07
bl_000  0.46  0.07
bk_000  0.39  0.06


Several sensors are missing for ~80% of healthy trucks but under 10% of failed ones, while ab_000 and cr_000 are 77% missing in both classes. Whether the missingness helps a model is tested in 02_baseline.